# Synthetic Evaluation Task Generation

Generate synthetic evaluation tasks for MCP tool-use benchmarks using
the sdg_hub distillation flow. Given an agent connected to MCP servers,
this notebook uses a frontier model to explore tools, generate grounded
questions, and produce expert-quality gold-standard trajectories.

### Pipeline

```
Your Agent + Frontier Model ──▶ Explore ──▶ Generate Questions ──▶ Expert Solves ──▶ benchmark_tasks.jsonl
```

The output (`benchmark_tasks.jsonl`) can be used with the companion
`evaluate.ipynb` notebook to benchmark different models through the same agent.


---
## 0. Setup

### 0.1 Prerequisites

**MCP servers**: Clone [mcp-bench](https://github.com/Accenture/mcp-bench) (provides the server source code) and start the servers:

```bash
git clone https://github.com/Accenture/mcp-bench.git ../mcp-bench
bash start_servers.sh          # installs deps + starts 6 servers on ports 8001-8006
bash start_servers.sh --check  # verify they're running
```

**LangGraph agents**: Each MCP server needs a LangGraph agent connected to it for the task generation step (Section 2).

1. Define a LangGraph graph that connects to your MCP server (e.g., a ReAct agent with MCP tools)
2. Serve it locally: `langgraph dev` (runs on `http://localhost:2024` by default)
3. For multiple servers, deploy one agent per server on different ports
4. Add each agent's URL to `.env`

**Note**: The exploration step may occasionally fail if the frontier model probes edge cases
that cause MCP tool errors (e.g., empty arrays, division by zero). The distillation flow has
built-in quality filtering that handles this — simply re-run the cell if a server fails.
A pre-generated `benchmark_tasks.jsonl` is provided so you can skip Section 2 entirely if needed.

**Environment**: Copy `.env.example` to `.env` and fill in your API key + LangGraph agent URLs.

In [ ]:
from pathlib import Path
import asyncio
import json
import os
import sys

from dotenv import load_dotenv
import nest_asyncio
import pandas as pd

nest_asyncio.apply()

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(NOTEBOOK_DIR / ".env")

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
assert OPENAI_API_KEY and OPENAI_API_KEY != "sk-...", "Set OPENAI_API_KEY in .env"

# Teacher model: the frontier model used for exploration + question generation
TEACHER_MODEL = os.environ.get("TEACHER_MODEL", "openai/gpt-5.2")

# LangGraph API key: optional authentication for LangGraph agent servers
LANGGRAPH_API_KEY = os.environ.get("LANGGRAPH_API_KEY", None)

# ── MCP Server URLs (for tool discovery only) ────────────────────────
# The distillation flow needs tool schemas as input. These URLs are used
# in Section 1 to list tools from each server. The servers are started
# by start_servers.sh and the agents connect to them automatically.
MCP_SERVERS = {
    "Weather Data": "http://localhost:8001/mcp",
    "Medical Calculator": "http://localhost:8002/mcp",
    "Wikipedia": "http://localhost:8003/mcp",
    "Car Price Evaluator": "http://localhost:8004/mcp",
    "Reddit": "http://localhost:8005/mcp",
    "DEX Paprika": "http://localhost:8006/mcp",
}

# ── LangGraph Agent URLs ─────────────────────────────────────────────
# Each agent is a LangGraph ReAct agent connected to one MCP server.
# Started by start_agents.sh on ports 2024-2029.
AGENT_URLS = {
    "Weather Data": os.environ.get(
        "LANGGRAPH_URL_WEATHER_DATA", "http://localhost:2024"
    ),
    "Medical Calculator": os.environ.get(
        "LANGGRAPH_URL_MEDICAL_CALCULATOR", "http://localhost:2025"
    ),
    "Wikipedia": os.environ.get("LANGGRAPH_URL_WIKIPEDIA", "http://localhost:2026"),
    "Car Price Evaluator": os.environ.get(
        "LANGGRAPH_URL_CAR_PRICE", "http://localhost:2027"
    ),
    "Reddit": os.environ.get("LANGGRAPH_URL_REDDIT", "http://localhost:2028"),
    "DEX Paprika": os.environ.get("LANGGRAPH_URL_DEX_PAPRIKA", "http://localhost:2029"),
}

print(f"Teacher model: {TEACHER_MODEL}")
print(f"Servers: {list(MCP_SERVERS.keys())}")
print(f"Agents:  {list(AGENT_URLS.keys())}")

---
## 1. Discover Tools

Connect to each MCP server and list its tools.

In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client


async def discover_tools(url):
    async with streamablehttp_client(url) as (r, w, _):
        async with ClientSession(r, w) as session:
            await session.initialize()
            resp = await session.list_tools()
            return [
                {
                    "name": t.name,
                    "description": t.description or "",
                    "inputSchema": t.inputSchema,
                }
                for t in resp.tools
            ]


all_tools = {}
for name, url in MCP_SERVERS.items():
    try:
        tools = asyncio.run(discover_tools(url))
        all_tools[name] = tools
        print(f"  {name}: {len(tools)} tools")
    except Exception as e:
        print(f"  {name}: FAILED — {e}")

assert all_tools, "No servers reachable! Start them with: bash start_servers.sh"
print(
    f"\nTotal: {sum(len(t) for t in all_tools.values())} tools across {len(all_tools)} servers"
)

---
## 2. Generate Evaluation Tasks

Run the MCP distillation flow on each server at varying complexity levels, then
transform the raw output into a clean evaluation dataset.

The `num_samples` parameter controls how many tools each generated question is designed around:
- `num_samples=2` → simpler questions using 2 tools
- `num_samples=4` → moderate questions using 4 tools
- `num_samples=8` → complex questions using 8 tools

Servers with fewer tools than `num_samples` are automatically skipped for that level.
All tasks are saved to a single `benchmark_tasks.jsonl` file with per-server caching —
servers that already have tasks in the file are skipped on re-run.

In [ ]:
from sdg_hub import Flow, FlowRegistry

FlowRegistry.discover_flows()

from eval_utils import extract_tool_names, normalize_tool_trace  # noqa: E402

# ── Configuration ─────────────────────────────────────────────────────
NUM_SAMPLES_LEVELS = [2, 4, 8]

SERVER_DESCRIPTIONS = {
    "Weather Data": "Weather data server providing current and forecast weather information.",
    "Medical Calculator": "Medical calculator server with 22 clinical calculation tools.",
    "Wikipedia": "Wikipedia server providing article search, content, and summarization.",
    "Car Price Evaluator": "Vehicle market pricing server for Brazilian car brands.",
    "Reddit": "Reddit server for fetching hot threads and post content.",
    "DEX Paprika": "DeFi/crypto analytics server with pool, token, and network data.",
}


def generate_tasks_for_server(server_name, tools, agent_url, num_samples=2):
    """Run the distillation flow for one server at a given num_samples level."""
    flow_instance = Flow.from_yaml(
        FlowRegistry.get_flow_path("MCP Server Distillation")
    )
    flow_instance.set_model_config(model=TEACHER_MODEL, api_key=OPENAI_API_KEY)

    agent_kwargs = {"agent_framework": "langgraph", "agent_url": agent_url}
    if LANGGRAPH_API_KEY:
        agent_kwargs["agent_api_key"] = LANGGRAPH_API_KEY
    flow_instance.set_agent_config(**agent_kwargs)
    flow_instance.set_agent_config(timeout=300, blocks=["explore_server"])
    flow_instance.set_agent_config(
        agent_framework="langgraph",
        blocks=["extract_exploration", "extract_agent_text"],
    )

    df = pd.DataFrame(
        {
            "tool_list": [tools],
            "mcp_server_name": [server_name],
            "mcp_server_description": [
                SERVER_DESCRIPTIONS.get(server_name, f"{server_name} MCP server")
            ],
        }
    )

    runtime_params = {}
    if num_samples != 2:
        runtime_params["sample_tools"] = {"num_samples": num_samples}

    result = flow_instance.generate(df, runtime_params=runtime_params)
    result_df = result.to_pandas() if hasattr(result, "to_pandas") else result

    export_cols = [
        c
        for c in [
            "question",
            "extract_agent_text_text",
            "extract_agent_text_tool_trace",
            "question_quality_rating",
            "completeness_rating",
        ]
        if c in result_df.columns
    ]
    return result_df[export_cols]


def transform_tasks(raw_df, server_name):
    """Transform raw distillation output into clean evaluation format."""
    df = raw_df.copy()
    df["expert_tool_trace"] = df["extract_agent_text_tool_trace"].apply(
        normalize_tool_trace
    )
    df["expert_tools"] = df["expert_tool_trace"].apply(extract_tool_names)
    df = df.rename(columns={"extract_agent_text_text": "expert_answer"})
    df["server"] = server_name
    return df[
        [
            "server",
            "question",
            "expert_answer",
            "expert_tools",
            "expert_tool_trace",
            "question_quality_rating",
            "completeness_rating",
        ]
    ]


print("Flow loaded: MCP Server Distillation")
print(f"Complexity levels: {NUM_SAMPLES_LEVELS}")
print(f"Servers: {list(all_tools.keys())}")

In [ ]:
BENCHMARK_PATH = NOTEBOOK_DIR / "benchmark_tasks.jsonl"

# Load existing benchmark (if any) for per-server caching
if BENCHMARK_PATH.exists():
    benchmark_df = pd.read_json(BENCHMARK_PATH, orient="records", lines=True)
    cached_servers = set(benchmark_df["server"].unique())
else:
    benchmark_df = pd.DataFrame()
    cached_servers = set()

new_tasks = []

for server_name, tools in all_tools.items():
    if server_name in cached_servers:
        n = len(benchmark_df[benchmark_df["server"] == server_name])
        print(f"{server_name}: {n} tasks (cached)")
        continue

    agent_url = AGENT_URLS.get(server_name, "")
    if not agent_url:
        print(f"{server_name}: SKIP — no agent URL configured")
        continue

    server_tasks = []
    n_tools = len(tools)

    for ns in NUM_SAMPLES_LEVELS:
        if n_tools < ns:
            print(f"  {server_name} ns={ns}: skipped ({n_tools} tools < {ns})")
            continue

        print(f"\n{'─' * 50}")
        print(f"{server_name} — num_samples={ns} ({n_tools} tools)")
        print(f"{'─' * 50}")

        try:
            result_df = generate_tasks_for_server(
                server_name, tools, agent_url, num_samples=ns
            )
            server_tasks.append(result_df)
            print(f"  Generated {len(result_df)} tasks")
        except Exception as e:
            print(f"  FAILED: {e}")

    if server_tasks:
        raw_combined = pd.concat(server_tasks, ignore_index=True)
        transformed = transform_tasks(raw_combined, server_name)
        new_tasks.append(transformed)
        print(f"\n  {server_name}: {len(transformed)} tasks")

if new_tasks:
    new_df = pd.concat(new_tasks, ignore_index=True)
    benchmark_df = pd.concat([benchmark_df, new_df], ignore_index=True)
    benchmark_df.to_json(BENCHMARK_PATH, orient="records", lines=True)
    print(f"\nAdded {len(new_df)} new tasks")

# Summary
print(
    f"\nBenchmark: {len(benchmark_df)} tasks across {benchmark_df['server'].nunique()} servers"
)
print(f"Columns: {list(benchmark_df.columns)}")
print("\nPer server:")
for server, group in benchmark_df.groupby("server"):
    print(f"  {server:<25} {len(group)} tasks")

---
## 3. Inspect Tasks

Sample task from the evaluation dataset.

In [ ]:
sample = benchmark_df.iloc[0]

print(f"Server: {sample['server']}")
print(f"\nQuestion:\n  {sample['question'][:400]}")
print(f"\nExpert Tools:\n  {sample['expert_tools']}")
print(f"\nQuality: {sample['question_quality_rating']}")
print(f"Completeness: {sample['completeness_rating']}")

print("\nExpert Trajectory:")
for i, step in enumerate(sample["expert_tool_trace"]):
    print(f"  [{i + 1}] {step['name']}({json.dumps(step.get('input', {}))[:80]})")

print(f"\nExpert Answer (first 300 chars):\n  {sample['expert_answer'][:300]}")